# Implementación

## Objetivo

* Implementar modelo de aprendizaje automático supervisado en un entorno de producción, garantizando su disponibilidad para realizar predicciones en tiempo real y clasificar si una página web presenta o no problemas de accesibilidad digital mediante una API (Interfaces de Programación de Aplicaciones).

## Funcionamiento

* **Entrada**: datos del formulario web: doc_language_ok, alt_texts_ok, input_labels_ok, empty_buttons_ok, empty_links_ok, color_contrast_ok, doc_language_fail, alt_texts_fail, input_labels_fail, empty_buttons_fail, empty_links_fail, color_contrast_fail
* **Proceso**: carga del modelo entrenado, captura de los datos del formulario y hacer la predicción 
* **Salida**: resultado de la predicción ('sin_prob_aw' / 'con_prob_aw')

## Formulario de entrada dentro del archivo index.html

In [1]:
%%file templates/index.html
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1" />
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.0.2/dist/css/bootstrap.min.css" rel="stylesheet" integrity="sha384-EVSTQN3/azprG1Anm3QDgpJLIm9Nao0Yz1ztcQTwFspd3yD65VohhpuuCOmLASjC" crossorigin="anonymous" />
    <title>Clasificación de páginas web según problemas de accesibilidad</title>
</head>

<body class="text-center"> 
    <h1 style = "color:#0d6efd;">Accesibilidad Web</h1> 
    <br><br>
    <form method="post" action="{{ url_for('predict') }}">
        <table class="table">
            <tr class="table-active">
                {% for var, val in results.items() %}
                    <th>{{ var }}</th>
                {% endfor %}
            </tr>
            <tr>
                {% for var, val in results.items() %}
                    <td><input type="number" name="{{ var }}" style="width:50px" value={{ val }} required></td>
                {% endfor %}
            </tr>
        </table>
        <button type="submit" class="btn btn-primary">Predecir</button>
    </form>
    <br><br>
    <footer>
        <h1>{{ prediction }}</h1>
    </footer>
</body>
</html>

Overwriting templates/index.html


## API con lógica de predicción

In [2]:
%%file api_prediction.py
from flask import Flask, request, render_template, jsonify
import pickle

# Cargar el modelo
with open("model.pkl", "rb") as f:
    model = pickle.load(f)

app = Flask(__name__)

variables = ['doc_language_ok', 'alt_texts_ok', 'input_labels_ok', 'empty_buttons_ok', 'empty_links_ok', 'color_contrast_ok', 
             'doc_language_fail', 'alt_texts_fail', 'input_labels_fail', 'empty_buttons_fail', 'empty_links_fail', 'color_contrast_fail']
data = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

@app.route("/")
def index():
    results = dict(zip(variables, data))
    return render_template("index.html", results=results)

@app.route('/predict', methods=['POST'])
def predict():
    data.clear()
    for var in variables:
        data.append(int(request.form.get(var)))
    prediction = model.predict([data])
    results = dict(zip(variables, data))
    return render_template('index.html', results=results, prediction=prediction[0])

Overwriting api_prediction.py


## Deploy en servidor de desarrollo

In [3]:
!export FLASK_APP=api_prediction.py

In [4]:
!flask --app api_prediction run

 * Serving Flask app 'api_prediction'
 * Debug mode: off
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [19/Dec/2025 16:49:00] "GET / HTTP/1.1" 200 -
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
127.0.0.1 - - [19/Dec/2025 16:49:10] "POST /predict HTTP/1.1" 200 -
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
127.0.0.1 - - [19/Dec/2025 16:49:14] "POST /predict HTTP/1.1" 200 -
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
127.0.0.1 - - [19/Dec/2025 16:50:05] "POST /predict HTTP/1.1" 200 -
/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: Us